# 🌶️ FOOD-SURE: Complete Backend Training & Comparison Pipeline

**Project:** Food-Sure — AI Based Food Adulteration Detection System  
**University:** Bahria University, Karachi Campus  
**Supervisor:** Dr. Asia Samreen  
**Team:** Mahnoor Shah · Ovais bin Amer · Nimra Rashid  

---

### Models Trained
| # | Model | Type |
|---|---|---|
| 1 | SVM (GLCM + Color Histogram) | Traditional ML |
| 2 | Random Forest (GLCM + Color Histogram) | Traditional ML |
| 3 | Custom CNN (Scratch) | Deep Learning |
| 4 | MobileNetV2 (Fine-tuned, 2-stage) | Transfer Learning |
| 5 | ResNet50 (Fine-tuned, 2-stage) | Transfer Learning |

### Spices
- Black Pepper Powder
- Red Chilli Powder
- Turmeric

### Classes (7 per spice)
Pure + 2 adulterants × 3 concentrations (5%, 10%, 15%)

## ⚙️ Cell 1 — Setup & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install split-folders scikit-image -q

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from skimage.feature import graycomatrix, graycoprops

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50  import preprocess_input as resnet_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import seaborn as sns
warnings.filterwarnings('ignore')

print('✅ All imports successful')
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 🔧 Cell 2 — Global Configuration

In [ ]:
BASE_DIR     = '/content/drive/MyDrive/Food Sure'
SPLIT_DIR    = os.path.join(BASE_DIR, 'dataset_split')
CKPT_DIR     = os.path.join(BASE_DIR, 'checkpoints_v2')
RESULTS_DIR  = os.path.join(BASE_DIR, 'results')

SPICES       = ['black_pepper_powder', 'red_chilli_powder', 'turmeric']
SPICE_LABELS = ['Black Pepper', 'Red Chilli', 'Turmeric']

SPLIT_RATIO  = (0.70, 0.15, 0.15)   # train / val / test
SEED         = 42
BATCH_SIZE   = 16
IMG_SIZE_ML  = 160    # SVM / RF feature extraction
IMG_SIZE_TL  = 224    # MobileNetV2 & ResNet50 (ImageNet standard)
IMG_SIZE_CNN = 128    # Custom CNN

os.makedirs(CKPT_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('✅ Configuration set')
print(f'   Base Dir    : {BASE_DIR}')
print(f'   Results Dir : {RESULTS_DIR}')
print(f'   Checkpoint  : {CKPT_DIR}')

## 📂 Cell 3 — Data Splitting (skips if already done)

In [ ]:
import splitfolders

for spice in SPICES:
    src = os.path.join(BASE_DIR, spice)
    dst = os.path.join(SPLIT_DIR, spice)
    if os.path.exists(dst):
        print(f'⏭️  Split already exists for {spice} — skipping.')
    else:
        print(f'✂️  Splitting {spice} ...')
        splitfolders.ratio(
            src, output=dst, seed=SEED,
            ratio=SPLIT_RATIO, group_prefix=None, move=False
        )

# Verify image counts per class per split
rows = []
for spice in SPICES:
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(SPLIT_DIR, spice, split)
        if not os.path.exists(split_path):
            continue
        for cls in sorted(os.listdir(split_path)):
            cls_path = os.path.join(split_path, cls)
            if os.path.isdir(cls_path):
                rows.append({
                    'spice': spice, 'split': split,
                    'class': cls,
                    'count': len(os.listdir(cls_path))
                })

df_counts = pd.DataFrame(rows)
pivot = df_counts.pivot_table(
    index=['spice', 'class'], columns='split',
    values='count', fill_value=0
)
print('\n📊 Image count per class per split:')
print(pivot.to_string())

## 🔬 Cell 4 — Feature Extraction: GLCM + Color Histogram

> As described in **Report Section 3.3 – Module 2**  
> Color features detect abnormal colour shifts; GLCM detects grain-size/roughness differences.

In [ ]:
def extract_hybrid_features(image):
    """
    Feature vector (101-dim) per image:
    - Color Histograms : 32 bins x 3 channels = 96 features
    - GLCM Texture     : 5 properties          =  5 features
    """
    # Color Histograms
    color_feats = []
    for ch in range(3):
        hist = cv2.calcHist([image], [ch], None, [32], [0, 256])
        cv2.normalize(hist, hist)
        color_feats.extend(hist.flatten())

    # GLCM Texture
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    glcm = graycomatrix(
        gray, distances=[1, 3],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256, symmetric=True, normed=True
    )
    texture_feats = [
        graycoprops(glcm, 'contrast').mean(),
        graycoprops(glcm, 'dissimilarity').mean(),
        graycoprops(glcm, 'homogeneity').mean(),
        graycoprops(glcm, 'energy').mean(),
        graycoprops(glcm, 'correlation').mean(),
    ]
    return np.array(color_feats + texture_feats)


def load_images_for_ml(spice, img_size=160):
    """
    Loads all images from the raw spice folder.
    Assigns 7-class labels directly from folder names.
    FIX: No longer uses binary 'pure'/'adulterated' keywords
         that caused the comparison.ipynb to skip all adulterated folders.
    """
    spice_dir = os.path.join(BASE_DIR, spice)
    class_names = sorted([
        d for d in os.listdir(spice_dir)
        if os.path.isdir(os.path.join(spice_dir, d))
        and d != 'dataset_split'
        and not d.startswith('.')
    ])
    class_to_idx = {cls: i for i, cls in enumerate(class_names)}

    X, y = [], []
    for cls in class_names:
        cls_dir = os.path.join(spice_dir, cls)
        for fname in os.listdir(cls_dir):
            fpath = os.path.join(cls_dir, fname)
            img = cv2.imread(fpath)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (img_size, img_size))
                X.append(img)
                y.append(class_to_idx[cls])

    print(f'  Loaded {len(X)} images | {len(class_names)} classes')
    print(f'  Class mapping: {class_to_idx}')
    return np.array(X), np.array(y), class_names


print('✅ Feature extraction functions defined')

## 🤖 Cell 5 — Traditional ML: SVM & Random Forest (7-class)

In [ ]:
ml_results = {}

for spice in SPICES:
    print(f"\n{'='*55}")
    print(f"  TRADITIONAL ML → {spice.upper().replace('_', ' ')}")
    print(f"{'='*55}")

    X_imgs, y, class_names = load_images_for_ml(spice, img_size=IMG_SIZE_ML)

    print('  Extracting GLCM + Color Histogram features ...')
    X_feats = np.array([extract_hybrid_features(img) for img in X_imgs])
    print(f'  Feature matrix shape: {X_feats.shape}')

    # Stratified split — all 7 classes guaranteed in train and test
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_feats, y, test_size=0.20, random_state=SEED, stratify=y
    )

    # StandardScaler — critical for SVM performance
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)

    # ── SVM ──────────────────────────────────────────────────
    print('\n  Training SVM (RBF kernel, C=10) ...')
    svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED)
    svm.fit(X_tr_sc, y_tr)
    svm_preds = svm.predict(X_te_sc)
    svm_acc   = accuracy_score(y_te, svm_preds) * 100
    print(f'  ✅ SVM Test Accuracy: {svm_acc:.2f}%')
    print(classification_report(y_te, svm_preds, target_names=class_names, zero_division=0))

    # ── Random Forest ─────────────────────────────────────────
    print('  Training Random Forest (200 trees) ...')
    rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
    rf.fit(X_tr_sc, y_tr)
    rf_preds = rf.predict(X_te_sc)
    rf_acc   = accuracy_score(y_te, rf_preds) * 100
    print(f'  ✅ Random Forest Test Accuracy: {rf_acc:.2f}%')
    print(classification_report(y_te, rf_preds, target_names=class_names, zero_division=0))

    ml_results[spice] = {
        'svm':         round(svm_acc, 2),
        'rf':          round(rf_acc,  2),
        'class_names': class_names
    }

with open(os.path.join(RESULTS_DIR, 'ml_results.json'), 'w') as f:
    json.dump(ml_results, f, indent=2)
print('\n✅ Traditional ML results saved to Drive.')

## 🧠 Cell 6 — Custom CNN (from scratch, with class weights)

In [ ]:
def build_custom_cnn(input_shape=(128, 128, 3), num_classes=7):
    """3-block CNN trained from scratch. Same architecture as FOOD-SURE.ipynb."""
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3,3), padding='same', activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        # Block 2
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        # Block 3
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        # Head
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model


def get_class_weights(generator):
    """Compute balanced class weights to prevent class collapse."""
    y = generator.classes
    weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
    return dict(enumerate(weights))


train_datagen_cnn = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    brightness_range=[0.8, 1.2],
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

cnn_results = {}

for spice in SPICES:
    print(f"\n{'='*55}")
    print(f"  CUSTOM CNN → {spice.upper().replace('_', ' ')}")
    print(f"{'='*55}")

    spice_split = os.path.join(SPLIT_DIR, spice)

    train_gen = train_datagen_cnn.flow_from_directory(
        os.path.join(spice_split, 'train'),
        target_size=(IMG_SIZE_CNN, IMG_SIZE_CNN),
        batch_size=BATCH_SIZE, class_mode='categorical',
        shuffle=True, seed=SEED
    )
    val_gen = val_test_datagen.flow_from_directory(
        os.path.join(spice_split, 'val'),
        target_size=(IMG_SIZE_CNN, IMG_SIZE_CNN),
        batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
    )
    test_gen = val_test_datagen.flow_from_directory(
        os.path.join(spice_split, 'test'),
        target_size=(IMG_SIZE_CNN, IMG_SIZE_CNN),
        batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
    )

    num_classes = train_gen.num_classes
    class_names = list(train_gen.class_indices.keys())
    class_w     = get_class_weights(train_gen)
    print(f'  Classes ({num_classes}): {class_names}')

    model = build_custom_cnn(
        input_shape=(IMG_SIZE_CNN, IMG_SIZE_CNN, 3), num_classes=num_classes
    )
    model.compile(
        optimizer=Adam(learning_rate=5e-4),
        loss='categorical_crossentropy', metrics=['accuracy']
    )

    ckpt_path = os.path.join(CKPT_DIR, f'{spice}_custom_cnn_best.keras')
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor='val_accuracy',
                        save_best_only=True, mode='max', verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=10,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=4, min_lr=1e-6, verbose=1)
    ]

    history = model.fit(
        train_gen, validation_data=val_gen, epochs=50,
        class_weight=class_w, callbacks=callbacks, verbose=1
    )

    test_gen.reset()
    _, test_acc = model.evaluate(test_gen, verbose=0)
    best_val    = max(history.history['val_accuracy'])
    print(f'\n  Best Val Accuracy : {best_val*100:.2f}%')
    print(f'  Test Accuracy     : {test_acc*100:.2f}%')

    cnn_results[spice] = {
        'test_acc':     round(test_acc * 100, 2),
        'best_val_acc': round(best_val  * 100, 2)
    }

with open(os.path.join(RESULTS_DIR, 'custom_cnn_results.json'), 'w') as f:
    json.dump(cnn_results, f, indent=2)
print('\n✅ Custom CNN results saved to Drive.')

## 🔧 Cell 7 — Transfer Learning Shared Helpers

In [ ]:
# NOTE: No rescale in these generators.
# preprocess_input inside each model handles normalisation correctly.
# Adding rescale=1/255 here AND preprocess_input inside the model
# would double-process the pixel values and hurt accuracy.

aug_datagen_tl   = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    brightness_range=[0.8, 1.2],
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)
plain_datagen_tl = ImageDataGenerator()  # no preprocessing — model handles it


def get_generators_tl(spice, img_size):
    spice_split = os.path.join(SPLIT_DIR, spice)
    train_gen = aug_datagen_tl.flow_from_directory(
        os.path.join(spice_split, 'train'),
        target_size=(img_size, img_size),
        batch_size=BATCH_SIZE, class_mode='categorical',
        shuffle=True, seed=SEED
    )
    val_gen = plain_datagen_tl.flow_from_directory(
        os.path.join(spice_split, 'val'),
        target_size=(img_size, img_size),
        batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
    )
    test_gen = plain_datagen_tl.flow_from_directory(
        os.path.join(spice_split, 'test'),
        target_size=(img_size, img_size),
        batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
    )
    return train_gen, val_gen, test_gen


def two_stage_train(model, base_model, train_gen, val_gen, test_gen,
                    spice, model_name, unfreeze_layers=30):
    """
    Two-stage fine-tuning:
      Stage 1 — Freeze entire base, train head only (lr=1e-3, 10 epochs)
      Stage 2 — Unfreeze top N layers, fine-tune end-to-end (lr=1e-5, 40 epochs)
    Class weights applied throughout to prevent class collapse.
    """
    class_names = list(train_gen.class_indices.keys())
    class_w     = get_class_weights(train_gen)
    print(f'  Classes: {class_names}')

    ckpt_path = os.path.join(CKPT_DIR, f'{spice}_{model_name}_best.keras')

    # ── Stage 1: Head only ──────────────────────────────────
    print(f'\n  Stage 1: Training head only (base frozen) ...')
    base_model.trainable = False
    model.compile(
        optimizer=Adam(1e-3),
        loss='categorical_crossentropy', metrics=['accuracy']
    )
    model.fit(
        train_gen, validation_data=val_gen, epochs=10,
        class_weight=class_w,
        callbacks=[
            EarlyStopping(monitor='val_accuracy', patience=5,
                          restore_best_weights=True, verbose=1)
        ],
        verbose=1
    )

    # ── Stage 2: Fine-tune top layers ───────────────────────
    print(f'\n  Stage 2: Fine-tuning top {unfreeze_layers} layers (lr=1e-5) ...')
    base_model.trainable = True
    for layer in base_model.layers[:-unfreeze_layers]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(1e-5),
        loss='categorical_crossentropy', metrics=['accuracy']
    )
    history = model.fit(
        train_gen, validation_data=val_gen, epochs=40,
        class_weight=class_w,
        callbacks=[
            ModelCheckpoint(ckpt_path, monitor='val_accuracy',
                            save_best_only=True, mode='max', verbose=1),
            EarlyStopping(monitor='val_accuracy', patience=10,
                          restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                              patience=4, min_lr=1e-7, verbose=1)
        ],
        verbose=1
    )

    test_gen.reset()
    _, test_acc = model.evaluate(test_gen, verbose=0)
    best_val    = max(history.history['val_accuracy'])
    print(f'\n  {model_name} | {spice}')
    print(f'  Best Val Accuracy : {best_val*100:.2f}%')
    print(f'  Test Accuracy     : {test_acc*100:.2f}%')

    return round(test_acc*100, 2), round(best_val*100, 2), class_names


print('✅ Transfer learning helpers defined')

## 📱 Cell 8 — MobileNetV2 (2-stage fine-tuning)

In [ ]:
def build_mobilenetv2(num_classes, img_size=224):
    base = MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False, weights='imagenet'
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = mobilenet_preprocess(inputs)    # [0,255] → [-1, 1]
    x       = base(x, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dropout(0.3)(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs), base


mobilenet_results = {}

for spice in SPICES:
    print(f"\n{'='*55}")
    print(f"  MOBILENETV2 → {spice.upper().replace('_', ' ')}")
    print(f"{'='*55}")

    train_gen, val_gen, test_gen = get_generators_tl(spice, IMG_SIZE_TL)
    num_classes = train_gen.num_classes

    model, base = build_mobilenetv2(num_classes, IMG_SIZE_TL)

    test_acc, best_val, class_names = two_stage_train(
        model, base, train_gen, val_gen, test_gen,
        spice, 'MobileNetV2', unfreeze_layers=30
    )
    mobilenet_results[spice] = {
        'test_acc':     test_acc,
        'best_val_acc': best_val,
        'class_names':  class_names
    }

with open(os.path.join(RESULTS_DIR, 'mobilenet_results.json'), 'w') as f:
    json.dump(mobilenet_results, f, indent=2)
print('\n✅ MobileNetV2 results saved to Drive.')

## 🏛️ Cell 9 — ResNet50 (2-stage fine-tuning)

In [ ]:
def build_resnet50(num_classes, img_size=224):
    base = ResNet50(
        input_shape=(img_size, img_size, 3),
        include_top=False, weights='imagenet'
    )
    base.trainable = False

    inputs  = tf.keras.Input(shape=(img_size, img_size, 3))
    x       = resnet_preprocess(inputs)       # [0,255] → zero-centered
    x       = base(x, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dropout(0.3)(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs), base


resnet_results = {}

for spice in SPICES:
    print(f"\n{'='*55}")
    print(f"  RESNET50 → {spice.upper().replace('_', ' ')}")
    print(f"{'='*55}")

    train_gen, val_gen, test_gen = get_generators_tl(spice, IMG_SIZE_TL)
    num_classes = train_gen.num_classes

    model, base = build_resnet50(num_classes, IMG_SIZE_TL)

    test_acc, best_val, class_names = two_stage_train(
        model, base, train_gen, val_gen, test_gen,
        spice, 'ResNet50', unfreeze_layers=30
    )
    resnet_results[spice] = {
        'test_acc':     test_acc,
        'best_val_acc': best_val,
        'class_names':  class_names
    }

with open(os.path.join(RESULTS_DIR, 'resnet_results.json'), 'w') as f:
    json.dump(resnet_results, f, indent=2)
print('\n✅ ResNet50 results saved to Drive.')

## 📥 Cell 10 — Load All Saved Results

In [ ]:
with open(os.path.join(RESULTS_DIR, 'ml_results.json'))         as f: ml_results       = json.load(f)
with open(os.path.join(RESULTS_DIR, 'custom_cnn_results.json')) as f: cnn_results       = json.load(f)
with open(os.path.join(RESULTS_DIR, 'mobilenet_results.json'))  as f: mobilenet_results = json.load(f)
with open(os.path.join(RESULTS_DIR, 'resnet_results.json'))     as f: resnet_results    = json.load(f)

spice_keys   = ['black_pepper_powder', 'red_chilli_powder', 'turmeric']
spice_labels = ['Black Pepper',        'Red Chilli',         'Turmeric']

svm_scores = [ml_results[s]['svm']             for s in spice_keys]
rf_scores  = [ml_results[s]['rf']              for s in spice_keys]
cnn_scores = [cnn_results[s]['test_acc']        for s in spice_keys]
mob_scores = [mobilenet_results[s]['test_acc']  for s in spice_keys]
res_scores = [resnet_results[s]['test_acc']     for s in spice_keys]

print('✅ All results loaded')
print(f'  SVM        : {svm_scores}')
print(f'  Rand Forest: {rf_scores}')
print(f'  Custom CNN : {cnn_scores}')
print(f'  MobileNetV2: {mob_scores}')
print(f'  ResNet50   : {res_scores}')

## 📊 Cell 11 — Master Comparison Bar Chart

In [ ]:
x = np.arange(len(spice_labels))
w = 0.15

fig, ax = plt.subplots(figsize=(14, 8))

b1 = ax.bar(x - 2*w, svm_scores, w, label='SVM (GLCM + Color)',           color='#1f77b4')
b2 = ax.bar(x -   w, rf_scores,  w, label='Random Forest (GLCM + Color)', color='#ff7f0e')
b3 = ax.bar(x,       cnn_scores, w, label='Custom CNN (Scratch)',          color='#d62728')
b4 = ax.bar(x +   w, mob_scores, w, label='MobileNetV2 (Fine-tuned)',     color='#2ca02c')
b5 = ax.bar(x + 2*w, res_scores, w, label='ResNet50 (Fine-tuned)',        color='#9467bd')

# Random-chance baseline for 7 classes
ax.axhline(y=14.29, color='gray', linestyle='--', linewidth=1.2,
           label='Random Baseline (14.3%)')

def label_bars(bars):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 4), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8.5, fontweight='bold')

for b in [b1, b2, b3, b4, b5]:
    label_bars(b)

ax.set_xlabel('Spice', fontsize=13)
ax.set_ylabel('Test Accuracy (%)', fontsize=13)
ax.set_title(
    'Food-Sure: Model Performance Comparison (Test Accuracy)\n'
    'SVM  |  Random Forest  |  Custom CNN  |  MobileNetV2  |  ResNet50',
    fontsize=14, fontweight='bold', pad=12
)
ax.set_xticks(x)
ax.set_xticklabels(spice_labels, fontsize=12)
ax.set_ylim(0, 110)
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
chart_path = os.path.join(RESULTS_DIR, 'model_comparison_chart.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Chart saved → {chart_path}')

## 📋 Cell 12 — Final Summary Table

In [ ]:
summary_rows = []
for label, key in zip(spice_labels, spice_keys):
    summary_rows.append({
        'Spice':         label,
        'SVM':           f"{ml_results[key]['svm']}%",
        'Random Forest': f"{ml_results[key]['rf']}%",
        'Custom CNN':    f"{cnn_results[key]['test_acc']}%",
        'MobileNetV2':   f"{mobilenet_results[key]['test_acc']}%",
        'ResNet50':      f"{resnet_results[key]['test_acc']}%",
    })

summary_df = pd.DataFrame(summary_rows).set_index('Spice')

print('\n' + '=' * 72)
print('   FOOD-SURE  —  FINAL MODEL COMPARISON  (Test Accuracy)')
print('=' * 72)
print(summary_df.to_string())
print('=' * 72)

print('\n  Best model per spice:')
for label, key in zip(spice_labels, spice_keys):
    scores = {
        'SVM':          ml_results[key]['svm'],
        'Random Forest':ml_results[key]['rf'],
        'Custom CNN':   cnn_results[key]['test_acc'],
        'MobileNetV2':  mobilenet_results[key]['test_acc'],
        'ResNet50':     resnet_results[key]['test_acc'],
    }
    best = max(scores, key=scores.get)
    print(f'    {label:<16} → {best} ({scores[best]:.2f}%)')

summary_df.to_csv(os.path.join(RESULTS_DIR, 'final_comparison_summary.csv'))
print(f'\n✅ Summary CSV saved → {RESULTS_DIR}')
print('\n🎉 FOOD-SURE BACKEND PIPELINE COMPLETE.')